In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, spearmanr
from scipy.stats.distributions import chi2
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import NMF
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split

# Load data
X0 = pd.read_excel('minmax01.xlsx').values
y = pd.read_csv('../idC_with_header.csv').values.flatten() - 1

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X0, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data split: Train {X_train.shape}, Test {X_test.shape}")

n_wavelengths = X_train.shape[1]
components_range0 = [10, 16, 20, 25, 32, 40, 50, 75, 100]

nmf_cache = {}

def fit_nmf_stable(X_train, X_test, n_comp, n_runs=10, n_seeds=None):
    """
    Fit NMF on training set only. Evaluate reconstruction error on both train & test.
    Returns U, V matrices, recon errors, and train-test metrics.
    """
    if n_seeds is None:
        n_seeds = np.arange(42, 42 + n_runs)
    
    U_train_list = []
    U_test_list = []
    V_list = []
    models = []
    recon_errors_train = []
    recon_errors_test = []
    
    for seed in n_seeds:
        nmf = NMF(n_components=n_comp, init='random', random_state=seed, max_iter=1000)
        
        # Fit on train
        U_train = nmf.fit_transform(X_train)
        U_train_list.append(U_train)
        V_list.append(nmf.components_)
        models.append(nmf)
        
        # Training reconstruction error
        recon_err_train = nmf.reconstruction_err_
        recon_errors_train.append(recon_err_train)
        
        # Test reconstruction error (transform test set, then evaluate)
        U_test = nmf.transform(X_test)
        X_test_recon = U_test @ nmf.components_
        recon_err_test = np.linalg.norm(X_test - X_test_recon, 'fro')
        recon_errors_test.append(recon_err_test)
        
        U_test_list.append(U_test)
    
    return {
        'U_train_list': U_train_list,
        'U_test_list': U_test_list,
        'V_list': V_list,
        'models': models,
        'recon_errors_train': recon_errors_train,
        'recon_errors_test': recon_errors_test,
        'recon_error_train_mean': np.mean(recon_errors_train),
        'recon_error_train_std': np.std(recon_errors_train),
        'recon_error_test_mean': np.mean(recon_errors_test),
        'recon_error_test_std': np.std(recon_errors_test),
        'generalization_gap': np.mean(recon_errors_test) - np.mean(recon_errors_train)
    }

print("="*70)
print("PHASE 1: FIT NMF ON TRAIN SET, EVALUATE ON TEST SET")
print("="*70)

for n_comp in [16]:
    print(f"\nFitting NMF with n_components={n_comp} across 10 seeds (train only)...")
    nmf_cache[n_comp] = fit_nmf_stable(X_train, X_test, n_comp, n_runs=10)
    
    cache_entry = nmf_cache[n_comp]
    print(f"  Train Recon Error: {cache_entry['recon_error_train_mean']:.4f} ± {cache_entry['recon_error_train_std']:.4f}")
    print(f"  Test Recon Error:  {cache_entry['recon_error_test_mean']:.4f} ± {cache_entry['recon_error_test_std']:.4f}")
    print(f"  Generalization Gap: {cache_entry['generalization_gap']:.4f}")

Data split: Train (354, 900), Test (89, 900)
PHASE 1: FIT NMF ON TRAIN SET, EVALUATE ON TEST SET

Fitting NMF with n_components=16 across 10 seeds (train only)...
  Train Recon Error: 42.9801 ± 0.0435
  Test Recon Error:  25.6152 ± 0.1647
  Generalization Gap: -17.3650


In [2]:
def analyze_factor_scores(U_list, y_train, factor_idx):
    """
    Kruskal-Wallis test on factor scores across classes.
    U_list: list of (n_samples, n_components) matrices from multiple NMF runs
    y_train: class labels (n_samples,)
    factor_idx: which factor to analyze
    """
    factor_scores = [U_list[i][:, factor_idx] for i in range(len(U_list))]
    factor_scores_mean = np.mean(factor_scores, axis=0)
    
    groups = [factor_scores_mean[y_train == c] for c in np.unique(y_train)]
    
    h_stat, p_value = kruskal(*groups)
    
    # Effect size (eta-squared approximation from H statistic)
    n = len(factor_scores_mean)
    k = len(groups)
    eta_sq = (h_stat - k + 1) / (n - k)
    eta_sq = max(0, eta_sq)  # Bound at 0
    
    # Spearman correlation with class (encode as ordinal)
    rho, rho_p = spearmanr(factor_scores_mean, y_train)
    
    return {
        'factor': factor_idx,
        'H_stat': h_stat,
        'p_value': p_value,
        'eta_sq': eta_sq,
        'spearman_rho': rho,
        'spearman_p': rho_p,
        'mean_score': np.mean(factor_scores_mean),
        'std_score': np.std(factor_scores_mean)
    }

def analyze_nmf_component(n_comp, U_train_list, V_list, y_train, X_train):
    """
    Full statistical analysis for one NMF component set.
    Uses training data only.
    """
    n_factors = U_train_list[0].shape[1]
    results = []
    
    for factor_idx in range(n_factors):
        factor_stats = analyze_factor_scores(U_train_list, y_train, factor_idx)
        results.append(factor_stats)
    
    df = pd.DataFrame(results)
    
    # BH correction
    reject, qvals, _, _ = multipletests(df['p_value'], method='fdr_bh')
    df['q_value_BH'] = qvals
    df['significant_BH'] = reject
    
    return df

print("\n" + "="*70)
print("PHASE 2: STATISTICAL TESTS ON FACTOR SCORES (U)")
print("="*70)

stats_by_component = {}

for n_comp in [16]:
    print(f"\nAnalyzing n_components={n_comp}...")
    U_train_list = nmf_cache[n_comp]['U_train_list']
    V_list = nmf_cache[n_comp]['V_list']
    
    df_stats = analyze_nmf_component(n_comp, U_train_list, V_list, y_train, X_train)
    stats_by_component[n_comp] = df_stats

    df_stats.to_csv(f"nmf_stats_{n_comp}.csv", index=False)

    print("\n" + "="*80)
    print(f"n_components = {n_comp}")
    print("="*80)
    
    df = stats_by_component[n_comp]
    print(df.to_string(index=False))
    
    # Show significant factors
    '''sig_factors = df_stats[df_stats['significant_BH']]
    print(f"  Significant factors (BH-adj): {len(sig_factors)}/{n_comp}")
    if len(sig_factors) > 0:
        print(sig_factors[['factor', 'H_stat', 'q_value_BH', 'eta_sq', 'spearman_rho']].to_string(index=False))'''


PHASE 2: STATISTICAL TESTS ON FACTOR SCORES (U)

Analyzing n_components=16...

n_components = 16
 factor     H_stat      p_value   eta_sq  spearman_rho   spearman_p  mean_score  std_score   q_value_BH  significant_BH
      0 209.608091 1.446156e-37 0.578259      0.155035 3.451806e-03    0.401631   0.175980 7.712833e-37            True
      1 127.926364 5.406459e-21 0.338019     -0.100997 5.764442e-02    0.328710   0.080784 9.611482e-21            True
      2 141.367310 1.119824e-23 0.377551      0.123453 2.015583e-02    0.233371   0.053723 3.583438e-23            True
      3 132.461788 6.759707e-22 0.351358      0.106041 4.618300e-02    0.204566   0.070687 1.545076e-21            True
      4  47.641554 7.526097e-06 0.101887      0.056767 2.868112e-01    0.185103   0.035009 7.526097e-06            True
      5 258.686287 1.002476e-47 0.722607     -0.218458 3.384780e-05    0.177279   0.097619 8.019808e-47            True
      6 148.367514 4.393864e-25 0.398140     -0.169779 1.34410

In [4]:
def analyze_basis_matrix(V_list, X0, top_n=5):
    """
    V: basis matrix (n_components, n_wavelengths)
    - Concentration: how peaked is each factor across wavelengths?
    - Top wavelengths: which wavelengths have highest weights?
    """
    wavenumbers = np.linspace(500, 4000, X0.shape[1])

    V_mean = np.mean(V_list, axis=0)  # Average across seeds
    n_components = V_mean.shape[0]
    #n_wavelengths = V_mean.shape[1]
    
    factor_info = []
    
    for factor_idx in range(n_components):
        basis = V_mean[factor_idx, :]
        
        # Concentration: Gini coefficient or entropy
        # Gini = 1 - (2 * sum(rank * weight)) / (n * sum(weight))
        sorted_basis = np.sort(basis)
        n = len(sorted_basis)
        gini = (2 * np.sum(np.arange(1, n+1) * sorted_basis)) / (n * np.sum(sorted_basis)) - (n + 1) / n
        
        # Top wavelengths
        top_indices = np.argsort(basis)[-top_n:][::-1]
        #top_wavelengths = wavenumbers[top_indices]
        top_wavelengths = np.round(wavenumbers[top_indices]).astype(int)
        top_weights = basis[top_indices]
        
        # Total mass in top N wavelengths
        top_mass_fraction = np.sum(top_weights) / np.sum(basis)

        dominant_wn = wavenumbers[top_indices[0]]
        if   900  <= dominant_wn < 1200: interp = "C-O / C-C stretch (sugars, polysaccharides)"
        elif 1200 <= dominant_wn < 1400: interp = "C-H bending / carboxylate stretch"
        elif 1400 <= dominant_wn < 1600: interp = "C=C / COO⁻ stretch (organic acids)"
        elif 1600 <= dominant_wn < 1700: interp = "Amide I / C=O stretch (proteins)"
        elif 1700 <= dominant_wn < 1800: interp = "C=O ester stretch (lipids, esters)"
        elif 2800 <= dominant_wn < 3000: interp = "C-H stretch (lipids, waxes)"
        elif 3000 <= dominant_wn <= 3600: interp = "O-H stretch (water, alcohols)"
        else:                             interp = "Mixed / overlapping bands"
        
        factor_info.append({
            'factor': factor_idx,
            'gini_concentration': gini,
            'top_mass_fraction': top_mass_fraction,
            'top_indices': top_indices,
            'top_wavelengths': top_wavelengths,
            'top_weights': top_weights,
            'max_weight': np.max(basis),
            'mean_weight': np.mean(basis),
            'interp': interp
        })

    df_basis = pd.DataFrame(factor_info)
    df_basis.to_csv(f"nmf_basis_interpreted_{n_comp}.csv", index=False)
    
    return factor_info

print("\n" + "="*70)
print("PHASE 3: BASIS MATRIX (V) - WAVELENGTH CONTRIBUTIONS")
print("="*70)

basis_by_component = {}

for n_comp in [16]:
    print(f"\nAnalyzing basis matrix for n_components={n_comp}...")
    V_list = nmf_cache[n_comp]['V_list']
    
    basis_info = analyze_basis_matrix(V_list, X0, top_n=5)
    basis_by_component[n_comp] = basis_info

    df_basis = pd.DataFrame(basis_info)
    df_basis.to_csv(f"NMF_basis_interpreted_{n_comp}.csv", index=False)
    
    '''for info in basis_info[:3]:  # Show first 3 factors
        print(f"  Factor {info['factor']}:")
        print(f"    Gini (concentration): {info['gini_concentration']:.3f}")
        print(f"    Top 5 wavelengths mass: {info['top_mass_fraction']:.2%}")
        print(f"    Top wavelengths: {info['top_wavelengths']}")
        print(f"    Top weights: {info['top_weights']}")'''

    for info in basis_by_component[n_comp]:
        
        print("\n----------------------------------")
        print(f"Factor {info['factor']}")
        print("----------------------------------")
        
        print("Gini:", info['gini_concentration'])
        print("Top mass fraction:", info['top_mass_fraction'])
        
        print("\nIndex | Wavelength | Weight")
        for idx, wl, w, inter in zip(info['top_indices'], info['top_wavelengths'], info['top_weights'], info['interp']):
            print(f"  {idx} | {wl:.2f} | {w:.2f} | {inter}")


PHASE 3: BASIS MATRIX (V) - WAVELENGTH CONTRIBUTIONS

Analyzing basis matrix for n_components=16...

----------------------------------
Factor 0
----------------------------------
Gini: 0.3297327706971327
Top mass fraction: 0.015141241059495806

Index | Wavelength | Weight
  712 | 3272.00 | 0.31 | O
  713 | 3276.00 | 0.30 | -
  714 | 3280.00 | 0.28 | H
  711 | 3268.00 | 0.28 |  
  715 | 3284.00 | 0.27 | s

----------------------------------
Factor 1
----------------------------------
Gini: 0.15472629195169008
Top mass fraction: 0.009202573183893559

Index | Wavelength | Weight
  365 | 1921.00 | 0.17 | M
  366 | 1925.00 | 0.16 | i
  711 | 3268.00 | 0.16 | x
  149 | 1080.00 | 0.16 | e
  173 | 1174.00 | 0.16 | d

----------------------------------
Factor 2
----------------------------------
Gini: 0.1471109409643554
Top mass fraction: 0.0096923529908195

Index | Wavelength | Weight
  806 | 3638.00 | 0.25 | M
  807 | 3642.00 | 0.24 | i
  695 | 3206.00 | 0.23 | x
  805 | 3634.00 | 0.23 | e


In [5]:
def bootstrap_stability(X, y, n_comp, n_bootstrap=50, seed=42):
    np.random.seed(seed)

    n_samples = X.shape[0]

    # store U matrices for each bootstrap
    U_bootstrap_list = []

    recon_errors = []

    for b in range(n_bootstrap):
        idx_boot = np.random.choice(n_samples, size=n_samples, replace=True)
        X_boot = X[idx_boot, :]

        nmf = NMF(n_components=n_comp, init='nndsvd', random_state=b, max_iter=1000)
        U = nmf.fit_transform(X_boot)

        recon_errors.append(nmf.reconstruction_err_)

        # reorder back to original space (approx alignment)
        U_bootstrap_list.append(U[np.argsort(idx_boot), :])

    U_bootstrap_list = np.array(U_bootstrap_list)  # shape: (B, n_samples, n_comp)

    # -------- FACTOR-WISE STABILITY --------
    factor_stability = []

    for k in range(n_comp):
        factor_values = U_bootstrap_list[:, :, k]  # all bootstraps for factor k

        # variability across bootstraps
        factor_std = np.std(factor_values)

        # coefficient of variation (normalized stability)
        factor_mean = np.mean(factor_values)
        factor_cv = factor_std / (factor_mean + 1e-8)

        factor_stability.append({
            'factor': k,
            'std': factor_std,
            'cv': factor_cv,
            'mean': factor_mean
        })

    return {
        'recon_error_mean': np.mean(recon_errors),
        'recon_error_std': np.std(recon_errors),
        'recon_error_cv': np.std(recon_errors) / np.mean(recon_errors),
        'factor_stability': factor_stability
    }

print("\n" + "="*70)
print("PHASE 4: STABILITY ACROSS BOOTSTRAP & CROSS-VALIDATION")
print("="*70)

stability_metrics = {}

for n_comp in [16]:
    boot = bootstrap_stability(X0, y, n_comp)

    df_factors = pd.DataFrame(boot['factor_stability'])
    df_factors.to_csv(f"nmf_stability_factors_{n_comp}.csv", index=False)
    
    print("\n" + "="*80)
    print(f"n_components = {n_comp}")
    print("="*80)

    print(f"Recon CV: {boot['recon_error_cv']:.4f}")

    print("\nFactor-wise stability:")
    for f in boot['factor_stability']:
        print(f"Factor {f['factor']:2d} | CV={f['cv']:.4f} | STD={f['std']:.4f}")


PHASE 4: STABILITY ACROSS BOOTSTRAP & CROSS-VALIDATION

n_components = 16
Recon CV: 0.0152

Factor-wise stability:
Factor  0 | CV=0.7145 | STD=0.2117
Factor  1 | CV=0.9919 | STD=0.2045
Factor  2 | CV=1.0635 | STD=0.2291
Factor  3 | CV=1.0295 | STD=0.2476
Factor  4 | CV=1.2482 | STD=0.3602
Factor  5 | CV=1.3638 | STD=0.2996
Factor  6 | CV=1.3038 | STD=0.2984
Factor  7 | CV=1.2511 | STD=0.2680
Factor  8 | CV=1.2962 | STD=0.3062
Factor  9 | CV=1.4576 | STD=0.4074
Factor 10 | CV=1.3329 | STD=0.3078
Factor 11 | CV=1.3111 | STD=0.3186
Factor 12 | CV=1.2809 | STD=0.2968
Factor 13 | CV=1.4803 | STD=0.2695
Factor 14 | CV=1.2801 | STD=0.2637
Factor 15 | CV=1.3360 | STD=0.3681


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kruskal, spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import NMF
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Load data
X0 = pd.read_excel('minmax01.xlsx').values
y = pd.read_csv('../idC_with_header.csv').values.flatten() - 1

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X0, y, test_size=0.2, random_state=42, stratify=y
)

components_range0 = [10, 16, 20, 25, 32, 40, 50, 75, 100]

# Results containers
def init_results():
    return {'n_components': [], 'test_acc_pct': [], 'test_f1': [], 'cv_f1': [],
            'cv_f1_std': [], 'n_sig_factors': [], 'mean_eta_sq': [], 'mean_rho': [],
            'bootstrap_cv': [], 'mean_gini': [], 'recon_error': []}

rf_results0 = init_results()
svm_results0 = init_results()
nbc_results0 = init_results()

# ===================== HELPERS =====================

def compute_stats(U_list, y_train, n_comp):
    results = []
    for factor_idx in range(n_comp):
        factor_scores = [U_list[i][:, factor_idx] for i in range(len(U_list))]
        factor_scores_mean = np.mean(factor_scores, axis=0)
        groups = [factor_scores_mean[y_train == c] for c in np.unique(y_train)]
        h_stat, p_value = kruskal(*groups)

        n = len(factor_scores_mean)
        k = len(groups)
        eta_sq = (h_stat - k + 1) / (n - k)
        eta_sq = max(0, eta_sq)

        rho, _ = spearmanr(factor_scores_mean, y_train)
        results.append({'p_value': p_value, 'eta_sq': eta_sq, 'rho': rho})

    df = pd.DataFrame(results)
    reject, qvals, _, _ = multipletests(df['p_value'], method='fdr_bh')
    df['significant'] = reject

    return df['significant'].sum(), df['eta_sq'].mean(), np.abs(df['rho']).mean()

def compute_bootstrap_cv(X_train, n_comp, n_bootstrap=10):
    errors = []
    for seed in range(42, 42 + n_bootstrap):
        idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
        X_boot = X_train[idx, :]
        nmf = NMF(n_components=n_comp, init='nndsvd', random_state=seed, max_iter=1000)
        nmf.fit_transform(X_boot)
        errors.append(nmf.reconstruction_err_)
    return np.std(errors) / np.mean(errors) if np.mean(errors) > 0 else 0

def compute_gini(V_list):
    V_mean = np.mean(V_list, axis=0)
    gini_scores = []
    for factor_idx in range(V_mean.shape[0]):
        basis = V_mean[factor_idx, :]
        sorted_basis = np.sort(basis)
        n = len(sorted_basis)
        gini = (2 * np.sum(np.arange(1, n+1) * sorted_basis)) / (n * np.sum(basis)) - (n + 1) / n
        gini_scores.append(gini)
    return np.mean(gini_scores)

# ===================== MAIN LOOP FUNCTION =====================

def run_experiment(results_dict, classifier, clf_name):
    print(f"Testing NMF with different n_components for {clf_name}")
    print("-" * 60)

    for n_comp in components_range0:

        # NMF (single, stable)
        nmf = NMF(n_components=n_comp, init='nndsvd', random_state=42, max_iter=1000)
        U_train = nmf.fit_transform(X_train)
        U_test = nmf.transform(X_test)
        V = nmf.components_

        # Train classifier
        clf = classifier
        clf.fit(U_train, y_train)

        # Test
        y_pred = clf.predict(U_test)
        test_acc = accuracy_score(y_test, y_pred)
        test_f1 = f1_score(y_test, y_pred, average='weighted')

        # Proper CV (no leakage)
        pipeline = Pipeline([
            ('nmf', NMF(n_components=n_comp, init='nndsvd', random_state=42, max_iter=1000)),
            ('clf', classifier)
        ])

        cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='f1_weighted')

        # Stats
        n_sig, mean_eta, mean_rho = compute_stats([U_train], y_train, n_comp)
        bootstrap_cv = compute_bootstrap_cv(X_train, n_comp)
        mean_gini = compute_gini([V])

        # Store
        results_dict['n_components'].append(n_comp)
        results_dict['test_acc_pct'].append(test_acc * 100)
        results_dict['test_f1'].append(test_f1)
        results_dict['cv_f1'].append(cv_scores.mean())
        results_dict['cv_f1_std'].append(cv_scores.std())
        results_dict['n_sig_factors'].append(n_sig)
        results_dict['mean_eta_sq'].append(mean_eta)
        results_dict['mean_rho'].append(mean_rho)
        results_dict['bootstrap_cv'].append(bootstrap_cv)
        results_dict['mean_gini'].append(mean_gini)
        results_dict['recon_error'].append(nmf.reconstruction_err_)

        print(f"n_components={n_comp:3d} | CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} | Test Acc: {test_acc*100:.2f}% | Sig Factors: {n_sig}/{n_comp}")

    best_n = results_dict['n_components'][np.argmax(results_dict['cv_f1'])]
    print(f"✓ Optimal for {clf_name}: {best_n}\n")
    return best_n

# ===================== RUN =====================

best_rf0 = run_experiment(rf_results0, RandomForestClassifier(random_state=42), "RF")
best_svm0 = run_experiment(svm_results0, SVC(random_state=42), "SVM")
best_nbc0 = run_experiment(nbc_results0, GaussianNB(), "NBC")

# ===================== PLOTTING =====================

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

def plot_results(ax_row, results, title, best_n):
    ax_row[0].plot(results['n_components'], results['recon_error'], 'bo-')
    ax_row[0].set_title(f'{title} - Reconstruction Error')
    ax_row[0].grid()

    ax_row[1].plot(results['n_components'], results['cv_f1'], 'go-')
    ax_row[1].axvline(best_n, color='r', linestyle='--')
    ax_row[1].set_title(f'{title} - F1 Score (Optimal: {best_n})')
    ax_row[1].grid()

plot_results(axes[0], rf_results0, "RF", best_rf0)
plot_results(axes[1], svm_results0, "SVM", best_svm0)
plot_results(axes[2], nbc_results0, "NBC", best_nbc0)

plt.tight_layout()
plt.show()

# ===================== SUMMARY =====================

print("\nSUMMARY")
print(f"RF Best: {best_rf0}")
print(f"SVM Best: {best_svm0}")
print(f"NBC Best: {best_nbc0}")

In [8]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import NMF
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
import warnings
warnings.filterwarnings('ignore')

# ===================== LOAD DATA =====================

X0 = pd.read_excel('minmax01.xlsx').values
y = pd.read_csv('../idC_with_header.csv').values.flatten() - 1

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X0, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Only 16 components
n_comp = 16

# ===================== RESULTS CONTAINER =====================

results = {
    'n_components': [],
    'test_acc_pct': [],
    'test_f1': [],
    'test_precision': [],
    'test_recall': [],
    'cv_f1': [],
    'cv_f1_std': [],
    'n_sig_factors': [],
    'mean_eta_sq': [],
    'mean_rho': [],
    'bootstrap_cv': [],
    'mean_gini': [],
    'recon_error': []
}

# ===================== HELPERS =====================

def compute_stats(U_list, y_train, n_comp):
    results_stats = []

    for factor_idx in range(n_comp):

        factor_scores = [U_list[i][:, factor_idx] for i in range(len(U_list))]
        factor_scores_mean = np.mean(factor_scores, axis=0)

        groups = [
            factor_scores_mean[y_train == c]
            for c in np.unique(y_train)
        ]

        h_stat, p_value = kruskal(*groups)

        n = len(factor_scores_mean)
        k = len(groups)

        eta_sq = (h_stat - k + 1) / (n - k)
        eta_sq = max(0, eta_sq)

        rho, _ = spearmanr(factor_scores_mean, y_train)

        results_stats.append({
            'p_value': p_value,
            'eta_sq': eta_sq,
            'rho': rho
        })

    df = pd.DataFrame(results_stats)

    reject, qvals, _, _ = multipletests(
        df['p_value'],
        method='fdr_bh'
    )

    df['significant'] = reject

    return (
        df['significant'].sum(),
        df['eta_sq'].mean(),
        np.abs(df['rho']).mean()
    )

def compute_bootstrap_cv(X_train, n_comp, n_bootstrap=10):

    errors = []

    for seed in range(42, 42 + n_bootstrap):

        idx = np.random.choice(
            len(X_train),
            size=len(X_train),
            replace=True
        )

        X_boot = X_train[idx, :]

        nmf = NMF(
            n_components=n_comp,
            init='nndsvd',
            random_state=seed,
            max_iter=1000
        )

        nmf.fit_transform(X_boot)

        errors.append(nmf.reconstruction_err_)

    return (
        np.std(errors) / np.mean(errors)
        if np.mean(errors) > 0 else 0
    )

def compute_gini(V_list):

    V_mean = np.mean(V_list, axis=0)

    gini_scores = []

    for factor_idx in range(V_mean.shape[0]):

        basis = V_mean[factor_idx, :]

        sorted_basis = np.sort(basis)

        n = len(sorted_basis)

        gini = (
            (2 * np.sum(np.arange(1, n + 1) * sorted_basis))
            / (n * np.sum(basis))
            - (n + 1) / n
        )

        gini_scores.append(gini)

    return np.mean(gini_scores)

# ===================== RUN SVM ONLY =====================

print("Testing SVM with NMF (16 components)")
print("-" * 60)

# NMF
nmf = NMF(
    n_components=n_comp,
    init='nndsvd',
    random_state=42,
    max_iter=1000
)

U_train = nmf.fit_transform(X_train)
U_test = nmf.transform(X_test)
V = nmf.components_

# SVM classifier
clf = SVC(random_state=42)

clf.fit(U_train, y_train)

# Predictions
y_pred = clf.predict(U_test)

# Metrics
test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='weighted')
test_precision = precision_score(y_test, y_pred, average='weighted')
test_recall = recall_score(y_test, y_pred, average='weighted')

# Cross-validation pipeline
pipeline = Pipeline([
    ('nmf', NMF(
        n_components=n_comp,
        init='nndsvd',
        random_state=42,
        max_iter=1000
    )),
    ('clf', SVC(random_state=42))
])

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

# Additional statistics
n_sig, mean_eta, mean_rho = compute_stats(
    [U_train],
    y_train,
    n_comp
)

bootstrap_cv = compute_bootstrap_cv(X_train, n_comp)
mean_gini = compute_gini([V])

# Store results
results['n_components'].append(n_comp)
results['test_acc_pct'].append(test_acc * 100)
results['test_f1'].append(test_f1)
results['test_precision'].append(test_precision)
results['test_recall'].append(test_recall)
results['cv_f1'].append(cv_scores.mean())
results['cv_f1_std'].append(cv_scores.std())
results['n_sig_factors'].append(n_sig)
results['mean_eta_sq'].append(mean_eta)
results['mean_rho'].append(mean_rho)
results['bootstrap_cv'].append(bootstrap_cv)
results['mean_gini'].append(mean_gini)
results['recon_error'].append(nmf.reconstruction_err_)

# ===================== OUTPUT =====================

print(f"""
Results for SVM with {n_comp} NMF components
--------------------------------------------------
Test Accuracy : {test_acc * 100:.2f}%
Test F1 Score : {test_f1:.4f}
Precision     : {test_precision:.4f}
Recall        : {test_recall:.4f}

CV F1 Score   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}

Significant Factors : {n_sig}/{n_comp}
Mean Eta Squared    : {mean_eta:.4f}
Mean Spearman Rho   : {mean_rho:.4f}
Bootstrap CV        : {bootstrap_cv:.4f}
Mean Gini           : {mean_gini:.4f}
Reconstruction Error: {nmf.reconstruction_err_:.4f}
""")

Testing SVM with NMF (16 components)
------------------------------------------------------------

Results for SVM with 16 NMF components
--------------------------------------------------
Test Accuracy : 92.13%
Test F1 Score : 0.9113
Precision     : 0.9228
Recall        : 0.9213

CV F1 Score   : 0.7634 ± 0.0344

Significant Factors : 16/16
Mean Eta Squared    : 0.3372
Mean Spearman Rho   : 0.1929
Bootstrap CV        : 0.0101
Mean Gini           : 0.4598
Reconstruction Error: 42.9949

